# Notebook 3: xarray & the Data Cube

Build a small multi-dimensional raster (data cube) with xarray and practice labeled selection.

**Dependencies:** `xarray`, `numpy`

In [ ]:
import numpy as np
import xarray as xr

## Create a 2D DataArray (single band)

Dimensions `y` and `x` with coordinates (e.g. northing, easting).

In [ ]:
ny, nx = 50, 80
y_coord = np.linspace(4000000, 3990000, ny)
x_coord = np.linspace(500000, 580000, nx)
data = np.random.rand(ny, nx)

da = xr.DataArray(
    data,
    dims=["y", "x"],
    coords={"y": y_coord, "x": x_coord},
    attrs={"long_name": "reflectance", "units": "1"},
)
da

## Select by coordinate (nearest)

`.sel()` uses coordinate values; `.isel()` uses integer indices.

In [ ]:
da.sel(x=540000, method="nearest")
# Slice in x and y
da.sel(x=slice(510000, 550000), y=slice(3995000, 3998000))

## Build a Dataset (multi-band)

Several variables (bands) sharing the same dimensions.

In [ ]:
red = np.random.rand(ny, nx).astype(np.float32)
nir = np.random.rand(ny, nx).astype(np.float32)

ds = xr.Dataset(
    {"red": (["y", "x"], red), "nir": (["y", "x"], nir)},
    coords={"y": y_coord, "x": x_coord},
)
ds

## Derive a new variable (NDVI)

Arithmetic is element-wise and respects dimensions.

In [ ]:
ds["ndvi"] = (ds["nir"] - ds["red"]) / (ds["nir"] + ds["red"])
ds["ndvi"]

## Add a time dimension (data cube)

Simulate 3 time steps; shape becomes (time, y, x).

In [ ]:
times = np.array(["2023-05-01", "2023-06-01", "2023-07-01"], dtype="datetime64[D]")
cube_data = np.random.rand(3, ny, nx).astype(np.float32)

cube = xr.DataArray(
    cube_data,
    dims=["time", "y", "x"],
    coords={"time": times, "y": y_coord, "x": x_coord},
)
cube.sel(time="2023-06-01")
cube.mean(dim=["y", "x"])  # one value per time

## Reduction along dimensions

`.mean()`, `.sum()`, `.max()` etc. can be applied along named dimensions.

In [ ]:
cube.mean(dim="time")  # (y, x)
cube.mean(dim=["y", "x"])  # (time,)